# [실습 2] 토크나이저 심화 — 특수 토큰, 패딩, 트런케이션, offset_mapping


## 실습 개요

> "텍스트를 숫자로 바꾸는 것 이상 — 토크나이저가 모델 성능에 미치는 영향은 무엇일까?"

토크나이저는 단순히 텍스트를 숫자 ID로 변환하는 것을 넘어,  
모델이 요구하는 입력 형식을 만드는 전처리 파이프라인 전체를 담당합니다.  

슬라이드에서 다룬 핵심 개념들을 코드로 직접 확인합니다:
- **특수 토큰** `[CLS]`, `[SEP]`, `[PAD]`, `[UNK]`의 역할과 실제 ID
- **패딩(Padding)**: 길이가 다른 문장들을 동일한 길이로 맞추는 방법
- **트런케이션(Truncation)**: 최대 길이를 초과하는 입력을 자르는 방법
- **attention_mask**: 패딩 토큰을 모델이 무시하도록 하는 마스크
- **offset_mapping**: 각 토큰이 원본 텍스트의 어느 위치에 대응하는지 추적
- **Fast vs Slow Tokenizer**: 속도와 기능 차이


## 실습 목표

1. 특수 토큰의 ID와 역할을 확인하고, 토크나이징 결과에서 위치를 찾는다.
2. 여러 문장을 배치로 토크나이징하며 패딩·트런케이션·attention_mask를 이해한다.
3. offset_mapping으로 토큰과 원본 텍스트 위치를 연결하는 방법을 익힌다.
4. [TODO] 배치 입력에 패딩과 트런케이션을 올바르게 적용하고 attention_mask를 확인한다.


## 목차

1. 라이브러리 임포트 및 토크나이저 로드
2. 특수 토큰 탐색
3. 기본 토크나이징 흐름 확인
4. 패딩(Padding)과 attention_mask
5. 트런케이션(Truncation)
6. offset_mapping으로 원본 위치 추적
7. [TODO] 배치 입력 전처리 직접 구성


---


## 1. 라이브러리 임포트 및 토크나이저 로드

`AutoTokenizer.from_pretrained()`는 허브에서 토크나이저 파일들을  
(`tokenizer_config.json`, `vocab.txt`, `special_tokens_map.json` 등) 내려받아 초기화합니다.  
use_fast=True(기본값)이면 Rust 기반 Fast Tokenizer를, False이면 Python Slow Tokenizer를 사용합니다.


In [ ]:
!pip install transformers
!python3.14 -m pip install --upgrade pip
!pip install torch

In [2]:
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer
import time

model_name = 'bert-base-uncased'

# use_fast=True: Rust 기반 Fast Tokenizer (기본값)
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
print(f'토크나이저 클래스: {type(tokenizer).__name__}')
print(f'어휘 사전 크기: {tokenizer.vocab_size:,} 토큰')


토크나이저 클래스: BertTokenizer
어휘 사전 크기: 30,522 토큰


## 2. 특수 토큰 탐색

BERT 계열 모델은 일반 단어 외에 특수 목적의 예약 토큰들을 가집니다:

| 토큰 | 이름 | 역할 |
|------|------|------|
| `[CLS]` | Classification | 문장 맨 앞에 삽입, 분류 태스크의 representation으로 사용 |
| `[SEP]` | Separator | 문장 끝 또는 두 문장 사이에 삽입하여 구분 |
| `[PAD]` | Padding | 배치 내 길이를 맞추기 위한 빈 토큰 (attention에서 무시됨) |
| `[UNK]` | Unknown | 어휘 사전에 없는 단어를 대체하는 토큰 |
| `[MASK]` | Mask | BERT 사전학습의 MLM 태스크에서 마스킹된 위치 |


In [3]:
# 특수 토큰 이름과 ID 출력
print('=== 특수 토큰 정보 ===')
special_tokens = {
    '[CLS]': tokenizer.cls_token_id,
    '[SEP]': tokenizer.sep_token_id,
    '[PAD]': tokenizer.pad_token_id,
    '[UNK]': tokenizer.unk_token_id,
    '[MASK]': tokenizer.mask_token_id,
}
for token, token_id in special_tokens.items():
    print(f'  {token} ID = {token_id}')
    #print(f'  {token:<10} ID = {token_id}')

# 모든 특수 토큰을 한 번에 확인
print(f'\n전체 특수 토큰: {tokenizer.all_special_tokens}')


=== 특수 토큰 정보 ===
  [CLS] ID = 101
  [SEP] ID = 102
  [PAD] ID = 0
  [UNK] ID = 100
  [MASK] ID = 103

전체 특수 토큰: ['[UNK]', '[SEP]', '[PAD]', '[CLS]', '[MASK]']


In [4]:
special_tokens.items()

dict_items([('[CLS]', 101), ('[SEP]', 102), ('[PAD]', 0), ('[UNK]', 100), ('[MASK]', 103)])

In [5]:
# 특수 토큰 ID를 다시 문자열로 디코딩
# convert_ids_to_tokens: ID 리스트 -> 토큰 문자열 리스트
ids = [tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id]
tokens = tokenizer.convert_ids_to_tokens(ids)
print(f'ID {ids} => 토큰 {tokens}')


ID [101, 102, 0] => 토큰 ['[CLS]', '[SEP]', '[PAD]']


## 3. 기본 토크나이징 흐름 확인

텍스트를 토크나이저에 넣으면 다음 단계를 거칩니다:  
**텍스트 입력 → 전처리(정규화) → 토큰 분절 → 특수 토큰 삽입 → ID 변환**  

`tokenizer(text)`의 반환값은 딕셔너리이며 주요 키는 다음과 같습니다:  
- `input_ids`: 토큰 ID 시퀀스 (`[CLS]` ID로 시작, `[SEP]` ID로 끝남)
- `attention_mask`: 실제 토큰 위치는 1, 패딩 위치는 0
- `token_type_ids`: 문장 A/B 구분 (BERT의 NSP 태스크용, 0 또는 1)


In [6]:
text = 'Hugging Face is building the future of NLP!'
encoding = tokenizer(text)

print(f'원본 텍스트: {text}\n')
print(f'input_ids      : {encoding["input_ids"]}')
print(f'attention_mask : {encoding["attention_mask"]}')
print(f'token_type_ids : {encoding["token_type_ids"]}\n')

# ID를 토큰 문자열로 역변환하여 분절 결과 확인
tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
print(f'토큰 (총 {len(tokens)}개):')
for i, (tok, id_) in enumerate(zip(tokens, encoding['input_ids'])):
    print(f'  [{i:2d}] {tok:<20} -> ID {id_}')


원본 텍스트: Hugging Face is building the future of NLP!

input_ids      : [101, 17662, 2227, 2003, 2311, 1996, 2925, 1997, 17953, 2361, 999, 102]
attention_mask : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
token_type_ids : [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

토큰 (총 12개):
  [ 0] [CLS]                -> ID 101
  [ 1] hugging              -> ID 17662
  [ 2] face                 -> ID 2227
  [ 3] is                   -> ID 2003
  [ 4] building             -> ID 2311
  [ 5] the                  -> ID 1996
  [ 6] future               -> ID 2925
  [ 7] of                   -> ID 1997
  [ 8] nl                   -> ID 17953
  [ 9] ##p                  -> ID 2361
  [10] !                    -> ID 999
  [11] [SEP]                -> ID 102


## 4. 패딩(Padding)과 attention_mask

GPU 연산은 배치 내 모든 시퀀스의 길이가 동일해야 합니다.  
길이가 다른 여러 문장을 배치로 처리하려면 짧은 문장 뒤에 `[PAD]` 토큰을 채워 길이를 맞춥니다.  

`padding` 옵션:
- `'max_length'`: `max_length` 인자로 지정한 길이까지 패딩
- `True` 또는 `'longest'`: 배치 내 가장 긴 시퀀스에 맞춰 패딩

패딩된 위치는 `attention_mask`에서 `0`으로 표시되어, 모델이 해당 위치를 어텐션 계산에서 제외합니다.


In [7]:
sentences = [
    'Hello.',                                    # 짧은 문장
    'Hugging Face is amazing.',                  # 중간 문장
    'Transformers library makes NLP much easier.', # 긴 문장
]

# padding='longest': 배치 내 가장 긴 문장에 맞춰 자동 패딩
batch = tokenizer(sentences, padding='longest', return_tensors='pt')

print(f'배치 input_ids shape: {tuple(batch["input_ids"].shape)}')
print(f'  (배치크기={batch["input_ids"].shape[0]}, 시퀀스길이={batch["input_ids"].shape[1]})')
print()

for i, sent in enumerate(sentences):
    ids = batch['input_ids'][i].tolist()
    mask = batch['attention_mask'][i].tolist()
    pad_count = ids.count(tokenizer.pad_token_id)
    print(f'문장 {i+1}: "{sent}"')
    print(f'  input_ids     : {ids}')
    print(f'  attention_mask: {mask}')
    print(f'  패딩 토큰 수: {pad_count}개\n')


배치 input_ids shape: (3, 10)
  (배치크기=3, 시퀀스길이=10)

문장 1: "Hello."
  input_ids     : [101, 7592, 1012, 102, 0, 0, 0, 0, 0, 0]
  attention_mask: [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]
  패딩 토큰 수: 6개

문장 2: "Hugging Face is amazing."
  input_ids     : [101, 17662, 2227, 2003, 6429, 1012, 102, 0, 0, 0]
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]
  패딩 토큰 수: 3개

문장 3: "Transformers library makes NLP much easier."
  input_ids     : [101, 19081, 3075, 3084, 17953, 2361, 2172, 6082, 1012, 102]
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
  패딩 토큰 수: 0개



## 5. 트런케이션(Truncation)

BERT의 최대 입력 길이는 512 토큰으로 제한됩니다.  
더 긴 텍스트를 입력하면 오류가 발생하므로, `truncation=True`를 지정하여 자동으로 잘라냅니다.  

`truncation` 옵션:  
- `True`: `max_length`까지 뒤쪽 토큰을 잘라냄 (기본값)  
- `'only_first'`: 두 문장 입력 시 첫 번째 문장만 자름  
- `'only_second'`: 두 문장 입력 시 두 번째 문장만 자름  

**항상 `[SEP]` 토큰은 마지막 위치에 보존됩니다.**


In [8]:
# 매우 긴 텍스트 생성 (512토큰 초과)
long_text = 'Hugging Face is amazing. ' * 50  # 반복하여 긴 텍스트 생성

# truncation=False이면 길이 제한 없이 토크나이징 (모델 입력 시 오류 발생 가능)
no_trunc = tokenizer(long_text, truncation=False)
print(f'트런케이션 없음: {len(no_trunc["input_ids"])} 토큰')

# truncation=True, max_length=64: 64 토큰으로 자름
with_trunc = tokenizer(long_text, truncation=True, max_length=64)
print(f'트런케이션 적용: {len(with_trunc["input_ids"])} 토큰')

# 마지막 토큰이 [SEP]인지 확인
last_token = tokenizer.convert_ids_to_tokens([with_trunc['input_ids'][-1]])[0]
print(f'마지막 토큰: {last_token} (항상 [SEP]로 끝남)')


트런케이션 없음: 252 토큰
트런케이션 적용: 64 토큰
마지막 토큰: [SEP] (항상 [SEP]로 끝남)


## 6. offset_mapping으로 원본 위치 추적

`return_offsets_mapping=True` 옵션을 주면 각 토큰이 원본 문자열에서  
**시작 인덱스(start)와 끝 인덱스(end)** 쌍으로 어느 위치를 차지하는지 반환합니다.  

예를 들어 `'hugging'`이 `'hug'`와 `'##ging'`으로 분절될 때,  
`offset_mapping`은 두 토큰 모두 원본 `'hugging'`(0~7)의 어느 부분인지 알려줍니다.  

이 정보는 개체명 인식(NER)이나 질의응답(QA) 등에서 모델 출력 토큰을  
원본 텍스트의 특정 단어·구절과 연결할 때 필수적으로 사용됩니다.


In [9]:
text = 'Hugging Face is building the future of NLP!'

# return_offsets_mapping=True: 각 토큰의 (start, end) 위치 반환
encoding = tokenizer(text, return_offsets_mapping=True)

tokens = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
offsets = encoding['offset_mapping']

print(f'원본 텍스트: "{text}"\n')
print(f'{"인덱스":<5} {"토큰":<15} {"오프셋":<12} {"원본 대응 문자"}')
print('-' * 55)
for i, (tok, (start, end)) in enumerate(zip(tokens, offsets)):
    # 특수 토큰([CLS], [SEP])은 (0, 0)으로 반환됨
    original_char = text[start:end] if end > start else '(특수토큰)'
    print(f'{i:<5} {tok:<15} ({start:>3},{end:>3})   "{original_char}"')


원본 텍스트: "Hugging Face is building the future of NLP!"

인덱스   토큰              오프셋          원본 대응 문자
-------------------------------------------------------
0     [CLS]           (  0,  0)   "(특수토큰)"
1     hugging         (  0,  7)   "Hugging"
2     face            (  8, 12)   "Face"
3     is              ( 13, 15)   "is"
4     building        ( 16, 24)   "building"
5     the             ( 25, 28)   "the"
6     future          ( 29, 35)   "future"
7     of              ( 36, 38)   "of"
8     nl              ( 39, 41)   "NL"
9     ##p             ( 41, 42)   "P"
10    !               ( 42, 43)   "!"
11    [SEP]           (  0,  0)   "(특수토큰)"


In [10]:
# offset_mapping 활용 예: 특정 단어 'building'의 토큰 위치 찾기
target_word = 'building'
target_start = text.find(target_word)
target_end = target_start + len(target_word)

print(f'찾는 단어: "{target_word}" (원본 위치: {target_start}~{target_end})')
print('\n해당 단어에 대응하는 토큰:')
for i, (tok, (start, end)) in enumerate(zip(tokens, offsets)):
    # 원본 텍스트에서 target 단어와 겹치는 토큰 탐색
    if start < target_end and end > target_start:
        print(f'  토큰 [{i}]: "{tok}" (오프셋 {start}~{end})')


찾는 단어: "building" (원본 위치: 16~24)

해당 단어에 대응하는 토큰:
  토큰 [4]: "building" (오프셋 16~24)


---
## 7. [TODO] 배치 입력 전처리 직접 구성

4개 문장을 하나의 배치로 토크나이징하며 패딩, 트런케이션, offset 정보를 함께 확인합니다.

TODO가 표시된 두 옵션은 배치 결과의 형태와 offset 반환 여부를 결정합니다.

- `return_tensors`에는 PyTorch 텐서를 의미하는 `'pt'`를 넣습니다.
- `return_offsets_mapping`에는 토큰별 원본 문자열 위치를 함께 받기 위해 `True`를 넣습니다.

나머지 옵션은 그대로 두면 배치 안의 가장 긴 문장 기준으로 패딩되고, 길이가 32 토큰을 넘는 입력은 잘립니다.


In [11]:
import torch

In [12]:
batch_sentences = [
    'Short sentence.',
    'A medium length sentence with more words than the first one.',
    'This is a much longer sentence that contains many words and might need truncation to fit.',
    'Another example.',
]

batch_encoding = tokenizer(
    batch_sentences,
    return_tensors="pt",  # TODO: PyTorch 텐서로 반환되도록 수정
    padding=True,
    truncation=True,
    max_length=32,
    return_offsets_mapping=True,  # TODO: 토큰의 원본 위치를 함께 반환하도록 수정
)

print(f'input_ids shape: {tuple(batch_encoding["input_ids"].shape)}')
print(f'attention_mask shape: {tuple(batch_encoding["attention_mask"].shape)}')
print()

print('문장별 attention_mask (1=실제토큰, 0=패딩):')
for i, sent in enumerate(batch_sentences):
    mask = batch_encoding['attention_mask'][i].tolist()
    real_tokens = sum(mask)
    pad_tokens = len(mask) - real_tokens
    print(f'  문장{i+1}: 실제 {real_tokens}개 + 패딩 {pad_tokens}개 = 총 {len(mask)}개')


input_ids shape: (4, 21)
attention_mask shape: (4, 21)

문장별 attention_mask (1=실제토큰, 0=패딩):
  문장1: 실제 5개 + 패딩 16개 = 총 21개
  문장2: 실제 14개 + 패딩 7개 = 총 21개
  문장3: 실제 21개 + 패딩 0개 = 총 21개
  문장4: 실제 5개 + 패딩 16개 = 총 21개
